# MI24 | Análise exploratória do inventário físico (PRD)

- **Fonte:** `prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico`.
- **Objeto:** view (indicado pelo prefixo `vw_`; confirme tipo e comentário na seção 1). **Ambiente:** PRD, somente leitura.
- **Schema do CSV:** 48 colunas: 38 string, 7 double, 2 decimal(13,3), 1 date. **Centro:** `cod_centro`. **Clustering:** não informado no CSV; conferir seção 1.
- **Como usar:** importar no Databricks como notebook SQL e executar Run All; cada célula lê a view diretamente. Registrar data/hora da execução. Os resultados não foram executados nesta entrega.

| Severidade | Alerta de schema / teste |
|---|---|
| 🔴 ALTA | `cod_usuario_contagem` e `cod_modificado`: apenas contagens e hashes nas amostras; nunca expor valor bruto. |
| 🟡 MÉDIA | Sete montantes `double`: conciliar com tolerância de 0,005; não interpretar diferença de ponto flutuante como divergência automática. |
| 🟡 MÉDIA | Datas `dt_*` são string: testar formato e sentinelas antes da comparação SAP. |
| 🟡 MÉDIA | Códigos com zeros à esquerda: medir colisões após normalização; não presumir que são números. |
| 🟡 MÉDIA | `num_referencia_inventario_fisico`: comentário declara NULL quando ausente; validar taxa de NULL/vazio. |
| ⚪ BAIXA | Chave real, PK declarada, clustering, comentário da view e filtros da tela SAP: não constam do CSV; validar antes do cenário. Nenhuma coluna foi declarada “sem fonte” no CSV. |

**Índice:** 1 Metadados · 2 Volumetria · 3 Centros (3.1 4128/4014) · 4 Granularidade · 5 Duplicidade · 6 Preenchimento (6.1 sem fonte) · 7 Cardinalidade · 8 Domínio · 9 Indicadores · 10 Perfil numérico · 11 Totais · 12 Datas · 13 Códigos · 14 Checksum · 15 Amostra · 16 Distribuição · 17 Freshness · 18 Regras MI24 · 19 Amostra ampliada · 20 Resumo.


## 1. Metadados
Inspecione o comentário da view, eventuais filtros SAP, PK declarada e propriedades. Como é uma view, DETAIL e HISTORY não fazem parte do Run All.

In [0]:
DESCRIBE EXTENDED prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico;

`DESCRIBE DETAIL` e `DESCRIBE HISTORY` não são executados em view. O CSV não contém o comentário da view nem clustering; se a seção 1 indicar que filtros da tela SAP não são aplicados, a comparação futura exigirá extração completa e equivalente à tela.

## 2. Volumetria
Estabelece tamanho e entidades distintas; distinção de chave é hipótese, não PK confirmada.

In [0]:
SELECT COUNT(*) AS linhas, COUNT(DISTINCT `cod_centro`) AS centros, COUNT(DISTINCT `cod_material`) AS materiais,
 COUNT(DISTINCT `num_inventario_fisico`) AS documentos,
 COUNT(DISTINCT `num_exercicio_fiscal`, `num_inventario_fisico`, `num_item_inventario_fisico`) AS itens_distintos_sem_nulos,
 COUNT_IF(`cod_centro` IS NULL OR trim(`cod_centro`) = '') AS centro_ausente
FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico;

## 3. Distribuição por centro
Centros entre 10 mil e 300 mil linhas são candidatos, não cenários aprovados.

In [0]:
SELECT `cod_centro`, COUNT(*) AS linhas, COUNT(DISTINCT `cod_material`) AS materiais,
 COUNT(DISTINCT `num_inventario_fisico`) AS documentos,
 CASE WHEN COUNT(*) BETWEEN 10000 AND 300000 THEN 'CANDIDATO A CENARIO' ELSE 'FORA DA FAIXA' END AS faixa
FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico GROUP BY `cod_centro` ORDER BY linhas DESC;

### 3.1. Centros 4128 e 4014
Confirma volume e formato do centro, sem pressupor que ambos existam.

In [0]:
SELECT `cod_centro`, COUNT(*) AS linhas, MIN(`dateingest`) AS primeira_ingestao, MAX(`dateingest`) AS ultima_ingestao
FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico WHERE trim(`cod_centro`) IN ('4128','4014') GROUP BY `cod_centro` ORDER BY `cod_centro`;

## 4. Granularidade
Compare cinco combinações candidatas. Distintos excluem NULL apenas quando a consulta explicitamente filtra; aqui `SELECT DISTINCT` inclui NULL como valor da combinação.

In [0]:
SELECT 'num_inventario_fisico' AS chave, COUNT(*) AS chaves_distintas, SUM(linhas) AS linhas, SUM(CASE WHEN linhas > 1 THEN linhas - 1 ELSE 0 END) AS linhas_excedentes, MAX(linhas) AS max_linhas_por_chave, ROUND(SUM(linhas) / NULLIF(COUNT(*),0),4) AS linhas_por_chave, SUM(CASE WHEN `num_inventario_fisico` IS NULL THEN linhas ELSE 0 END) AS linhas_com_chave_nula FROM (SELECT `num_inventario_fisico`, COUNT(*) AS linhas FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico GROUP BY `num_inventario_fisico`) k
UNION ALL
SELECT 'num_exercicio_fiscal + num_inventario_fisico' AS chave, COUNT(*) AS chaves_distintas, SUM(linhas) AS linhas, SUM(CASE WHEN linhas > 1 THEN linhas - 1 ELSE 0 END) AS linhas_excedentes, MAX(linhas) AS max_linhas_por_chave, ROUND(SUM(linhas) / NULLIF(COUNT(*),0),4) AS linhas_por_chave, SUM(CASE WHEN `num_exercicio_fiscal` IS NULL OR `num_inventario_fisico` IS NULL THEN linhas ELSE 0 END) AS linhas_com_chave_nula FROM (SELECT `num_exercicio_fiscal`, `num_inventario_fisico`, COUNT(*) AS linhas FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico GROUP BY `num_exercicio_fiscal`, `num_inventario_fisico`) k
UNION ALL
SELECT 'num_exercicio_fiscal + num_inventario_fisico + num_item_inventario_fisico' AS chave, COUNT(*) AS chaves_distintas, SUM(linhas) AS linhas, SUM(CASE WHEN linhas > 1 THEN linhas - 1 ELSE 0 END) AS linhas_excedentes, MAX(linhas) AS max_linhas_por_chave, ROUND(SUM(linhas) / NULLIF(COUNT(*),0),4) AS linhas_por_chave, SUM(CASE WHEN `num_exercicio_fiscal` IS NULL OR `num_inventario_fisico` IS NULL OR `num_item_inventario_fisico` IS NULL THEN linhas ELSE 0 END) AS linhas_com_chave_nula FROM (SELECT `num_exercicio_fiscal`, `num_inventario_fisico`, `num_item_inventario_fisico`, COUNT(*) AS linhas FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico GROUP BY `num_exercicio_fiscal`, `num_inventario_fisico`, `num_item_inventario_fisico`) k
UNION ALL
SELECT 'cod_centro + num_exercicio_fiscal + num_inventario_fisico + num_item_inventario_fisico' AS chave, COUNT(*) AS chaves_distintas, SUM(linhas) AS linhas, SUM(CASE WHEN linhas > 1 THEN linhas - 1 ELSE 0 END) AS linhas_excedentes, MAX(linhas) AS max_linhas_por_chave, ROUND(SUM(linhas) / NULLIF(COUNT(*),0),4) AS linhas_por_chave, SUM(CASE WHEN `cod_centro` IS NULL OR `num_exercicio_fiscal` IS NULL OR `num_inventario_fisico` IS NULL OR `num_item_inventario_fisico` IS NULL THEN linhas ELSE 0 END) AS linhas_com_chave_nula FROM (SELECT `cod_centro`, `num_exercicio_fiscal`, `num_inventario_fisico`, `num_item_inventario_fisico`, COUNT(*) AS linhas FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico GROUP BY `cod_centro`, `num_exercicio_fiscal`, `num_inventario_fisico`, `num_item_inventario_fisico`) k
UNION ALL
SELECT 'cod_centro + num_exercicio_fiscal + num_inventario_fisico + num_item_inventario_fisico + cod_deposito' AS chave, COUNT(*) AS chaves_distintas, SUM(linhas) AS linhas, SUM(CASE WHEN linhas > 1 THEN linhas - 1 ELSE 0 END) AS linhas_excedentes, MAX(linhas) AS max_linhas_por_chave, ROUND(SUM(linhas) / NULLIF(COUNT(*),0),4) AS linhas_por_chave, SUM(CASE WHEN `cod_centro` IS NULL OR `num_exercicio_fiscal` IS NULL OR `num_inventario_fisico` IS NULL OR `num_item_inventario_fisico` IS NULL OR `cod_deposito` IS NULL THEN linhas ELSE 0 END) AS linhas_com_chave_nula FROM (SELECT `cod_centro`, `num_exercicio_fiscal`, `num_inventario_fisico`, `num_item_inventario_fisico`, `cod_deposito`, COUNT(*) AS linhas FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico GROUP BY `cod_centro`, `num_exercicio_fiscal`, `num_inventario_fisico`, `num_item_inventario_fisico`, `cod_deposito`) k;

## 5. Duplicidade
Separe linhas inteiramente idênticas de versões diferentes na chave candidata; as colunas variáveis são contadas apenas nos grupos duplicados. Não declare chave real antes de avaliar a seção 4.

In [0]:
WITH agrupado AS (SELECT `cod_centro`, `num_exercicio_fiscal`, `num_inventario_fisico`, `num_item_inventario_fisico`, COUNT(*) AS linhas, COUNT(DISTINCT md5(to_json(named_struct('num_inventario_fisico', `num_inventario_fisico`, 'num_item_inventario_fisico', `num_item_inventario_fisico`, 'num_exercicio_fiscal', `num_exercicio_fiscal`, 'cod_material', `cod_material`, 'desc_material', `desc_material`, 'tp_categoria_material', `tp_categoria_material`, 'num_lote', `num_lote`, 'cod_centro', `cod_centro`, 'cod_deposito', `cod_deposito`, 'ind_item_contado', `ind_item_contado`, 'cod_diferenca_item_esta_registrada', `cod_diferenca_item_esta_registrada`, 'ind_item_recontado', `ind_item_recontado`, 'ind_item_eliminado', `ind_item_eliminado`, 'cod_contagem_zero', `cod_contagem_zero`, 'tp_estoque_inventario_fisico', `tp_estoque_inventario_fisico`, 'tp_estoque_especial', `tp_estoque_especial`, 'tp_inventario_rotativo', `tp_inventario_rotativo`, 'vl_quantidade_contagem_inventario', `vl_quantidade_contagem_inventario`, 'vl_diferenca_inventario_entrada_contagem', `vl_diferenca_inventario_entrada_contagem`, 'vl_precos_venda_excluindo_iva', `vl_precos_venda_excluindo_iva`, 'vl_precos_venda_iva', `vl_precos_venda_iva`, 'vl_inserido_externamente_base_venda', `vl_inserido_externamente_base_venda`, 'cod_cliente', `cod_cliente`, 'cod_fornecedor', `cod_fornecedor`, 'cod_ordem_cliente', `cod_ordem_cliente`, 'num_item_pedido_venda', `num_item_pedido_venda`, 'num_divisao_remessa', `num_divisao_remessa`, 'ind_material_valor', `ind_material_valor`, 'qt_registrada_antes_contagem', `qt_registrada_antes_contagem`, 'qt_quantidade', `qt_quantidade`, 'sg_medida_basica_material', `sg_medida_basica_material`, 'vl_contabil', `vl_contabil`, 'vl_diferenca', `vl_diferenca`, 'cod_moeda', `cod_moeda`, 'dt_lancamento', `dt_lancamento`, 'dt_realizacao_ultima_contagem', `dt_realizacao_ultima_contagem`, 'dt_ultima_modificacao', `dt_ultima_modificacao`, 'cod_usuario_contagem', `cod_usuario_contagem`, 'cod_modificado', `cod_modificado`, 'num_referencia_inventario_fisico', `num_referencia_inventario_fisico`, 'num_material', `num_material`, 'num_ano_material', `num_ano_material`, 'num_item_material', `num_item_material`, 'num_recontagem', `num_recontagem`, 'cod_motivo_diferenca', `cod_motivo_diferenca`, 'dateingest', `dateingest`, 'yearingest', `yearingest`, 'monthingest', `monthingest`)))) AS versoes FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico GROUP BY `cod_centro`, `num_exercicio_fiscal`, `num_inventario_fisico`, `num_item_inventario_fisico`)
SELECT COUNT_IF(linhas > 1) AS chaves_repetidas, COALESCE(SUM(CASE WHEN linhas > 1 THEN linhas ELSE 0 END),0) AS linhas_em_chaves_repetidas,
 COALESCE(SUM(CASE WHEN linhas > 1 THEN linhas - versoes ELSE 0 END),0) AS repeticoes_identicas,
 COALESCE(SUM(CASE WHEN linhas > 1 THEN versoes - 1 ELSE 0 END),0) AS versoes_adicionais,
 MAX(linhas) AS max_linhas_por_chave FROM agrupado;

In [0]:
WITH d AS (SELECT `cod_centro`, `num_exercicio_fiscal`, `num_inventario_fisico`, `num_item_inventario_fisico`, COUNT(*) AS linhas,
 COUNT(DISTINCT COALESCE(CAST(`cod_material` AS STRING), '<NULL>')) AS v_cod_material, COUNT(DISTINCT COALESCE(CAST(`cod_deposito` AS STRING), '<NULL>')) AS v_cod_deposito, COUNT(DISTINCT COALESCE(CAST(`num_lote` AS STRING), '<NULL>')) AS v_num_lote, COUNT(DISTINCT COALESCE(CAST(`qt_quantidade` AS STRING), '<NULL>')) AS v_qt_quantidade, COUNT(DISTINCT COALESCE(CAST(`qt_registrada_antes_contagem` AS STRING), '<NULL>')) AS v_qt_registrada_antes_contagem, COUNT(DISTINCT COALESCE(CAST(`vl_diferenca` AS STRING), '<NULL>')) AS v_vl_diferenca, COUNT(DISTINCT COALESCE(CAST(`dateingest` AS STRING), '<NULL>')) AS v_dateingest, COUNT(DISTINCT COALESCE(CAST(`num_recontagem` AS STRING), '<NULL>')) AS v_num_recontagem, COUNT(DISTINCT COALESCE(CAST(`num_material` AS STRING), '<NULL>')) AS v_num_material
 FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico GROUP BY `cod_centro`, `num_exercicio_fiscal`, `num_inventario_fisico`, `num_item_inventario_fisico`)
SELECT `cod_centro`, `num_exercicio_fiscal`, `num_inventario_fisico`, `num_item_inventario_fisico`, linhas, v_cod_material, v_cod_deposito, v_num_lote, v_qt_quantidade, v_qt_registrada_antes_contagem, v_vl_diferenca, v_dateingest, v_num_recontagem, v_num_material
FROM d WHERE linhas > 1 ORDER BY linhas DESC LIMIT 100;

## 6. Preenchimento de todas as colunas
Nulo, vazio/sentinela textual, zero e útil por coluna. Em indicadores, vazio pode significar “não”, não erro. Zero numérico é valor legítimo a investigar.

In [0]:
WITH t AS (SELECT COUNT(*) AS total FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico),
 agg AS (SELECT COUNT_IF(`num_inventario_fisico` IS NULL) AS n_0, COUNT_IF(`num_inventario_fisico` IS NOT NULL AND lower(trim(`num_inventario_fisico`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_0, COUNT_IF(`num_inventario_fisico` IS NOT NULL AND trim(`num_inventario_fisico`) RLIKE '^0+([.,]0+)?$') AS z_0, COUNT_IF(`num_item_inventario_fisico` IS NULL) AS n_1, COUNT_IF(`num_item_inventario_fisico` IS NOT NULL AND lower(trim(`num_item_inventario_fisico`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_1, COUNT_IF(`num_item_inventario_fisico` IS NOT NULL AND trim(`num_item_inventario_fisico`) RLIKE '^0+([.,]0+)?$') AS z_1, COUNT_IF(`num_exercicio_fiscal` IS NULL) AS n_2, COUNT_IF(`num_exercicio_fiscal` IS NOT NULL AND lower(trim(`num_exercicio_fiscal`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_2, COUNT_IF(`num_exercicio_fiscal` IS NOT NULL AND trim(`num_exercicio_fiscal`) RLIKE '^0+([.,]0+)?$') AS z_2, COUNT_IF(`cod_material` IS NULL) AS n_3, COUNT_IF(`cod_material` IS NOT NULL AND lower(trim(`cod_material`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_3, COUNT_IF(`cod_material` IS NOT NULL AND trim(`cod_material`) RLIKE '^0+([.,]0+)?$') AS z_3, COUNT_IF(`desc_material` IS NULL) AS n_4, COUNT_IF(`desc_material` IS NOT NULL AND lower(trim(`desc_material`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_4, COUNT_IF(`desc_material` IS NOT NULL AND trim(`desc_material`) RLIKE '^0+([.,]0+)?$') AS z_4, COUNT_IF(`tp_categoria_material` IS NULL) AS n_5, COUNT_IF(`tp_categoria_material` IS NOT NULL AND lower(trim(`tp_categoria_material`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_5, COUNT_IF(`tp_categoria_material` IS NOT NULL AND trim(`tp_categoria_material`) RLIKE '^0+([.,]0+)?$') AS z_5, COUNT_IF(`num_lote` IS NULL) AS n_6, COUNT_IF(`num_lote` IS NOT NULL AND lower(trim(`num_lote`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_6, COUNT_IF(`num_lote` IS NOT NULL AND trim(`num_lote`) RLIKE '^0+([.,]0+)?$') AS z_6, COUNT_IF(`cod_centro` IS NULL) AS n_7, COUNT_IF(`cod_centro` IS NOT NULL AND lower(trim(`cod_centro`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_7, COUNT_IF(`cod_centro` IS NOT NULL AND trim(`cod_centro`) RLIKE '^0+([.,]0+)?$') AS z_7, COUNT_IF(`cod_deposito` IS NULL) AS n_8, COUNT_IF(`cod_deposito` IS NOT NULL AND lower(trim(`cod_deposito`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_8, COUNT_IF(`cod_deposito` IS NOT NULL AND trim(`cod_deposito`) RLIKE '^0+([.,]0+)?$') AS z_8, COUNT_IF(`ind_item_contado` IS NULL) AS n_9, COUNT_IF(`ind_item_contado` IS NOT NULL AND lower(trim(`ind_item_contado`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_9, COUNT_IF(`ind_item_contado` IS NOT NULL AND trim(`ind_item_contado`) RLIKE '^0+([.,]0+)?$') AS z_9, COUNT_IF(`cod_diferenca_item_esta_registrada` IS NULL) AS n_10, COUNT_IF(`cod_diferenca_item_esta_registrada` IS NOT NULL AND lower(trim(`cod_diferenca_item_esta_registrada`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_10, COUNT_IF(`cod_diferenca_item_esta_registrada` IS NOT NULL AND trim(`cod_diferenca_item_esta_registrada`) RLIKE '^0+([.,]0+)?$') AS z_10, COUNT_IF(`ind_item_recontado` IS NULL) AS n_11, COUNT_IF(`ind_item_recontado` IS NOT NULL AND lower(trim(`ind_item_recontado`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_11, COUNT_IF(`ind_item_recontado` IS NOT NULL AND trim(`ind_item_recontado`) RLIKE '^0+([.,]0+)?$') AS z_11, COUNT_IF(`ind_item_eliminado` IS NULL) AS n_12, COUNT_IF(`ind_item_eliminado` IS NOT NULL AND lower(trim(`ind_item_eliminado`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_12, COUNT_IF(`ind_item_eliminado` IS NOT NULL AND trim(`ind_item_eliminado`) RLIKE '^0+([.,]0+)?$') AS z_12, COUNT_IF(`cod_contagem_zero` IS NULL) AS n_13, COUNT_IF(`cod_contagem_zero` IS NOT NULL AND lower(trim(`cod_contagem_zero`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_13, COUNT_IF(`cod_contagem_zero` IS NOT NULL AND trim(`cod_contagem_zero`) RLIKE '^0+([.,]0+)?$') AS z_13, COUNT_IF(`tp_estoque_inventario_fisico` IS NULL) AS n_14, COUNT_IF(`tp_estoque_inventario_fisico` IS NOT NULL AND lower(trim(`tp_estoque_inventario_fisico`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_14, COUNT_IF(`tp_estoque_inventario_fisico` IS NOT NULL AND trim(`tp_estoque_inventario_fisico`) RLIKE '^0+([.,]0+)?$') AS z_14, COUNT_IF(`tp_estoque_especial` IS NULL) AS n_15, COUNT_IF(`tp_estoque_especial` IS NOT NULL AND lower(trim(`tp_estoque_especial`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_15, COUNT_IF(`tp_estoque_especial` IS NOT NULL AND trim(`tp_estoque_especial`) RLIKE '^0+([.,]0+)?$') AS z_15, COUNT_IF(`tp_inventario_rotativo` IS NULL) AS n_16, COUNT_IF(`tp_inventario_rotativo` IS NOT NULL AND lower(trim(`tp_inventario_rotativo`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_16, COUNT_IF(`tp_inventario_rotativo` IS NOT NULL AND trim(`tp_inventario_rotativo`) RLIKE '^0+([.,]0+)?$') AS z_16, COUNT_IF(`vl_quantidade_contagem_inventario` IS NULL) AS n_17, CAST(0 AS BIGINT) AS e_17, COUNT_IF(`vl_quantidade_contagem_inventario` = 0) AS z_17, COUNT_IF(`vl_diferenca_inventario_entrada_contagem` IS NULL) AS n_18, CAST(0 AS BIGINT) AS e_18, COUNT_IF(`vl_diferenca_inventario_entrada_contagem` = 0) AS z_18, COUNT_IF(`vl_precos_venda_excluindo_iva` IS NULL) AS n_19, CAST(0 AS BIGINT) AS e_19, COUNT_IF(`vl_precos_venda_excluindo_iva` = 0) AS z_19, COUNT_IF(`vl_precos_venda_iva` IS NULL) AS n_20, CAST(0 AS BIGINT) AS e_20, COUNT_IF(`vl_precos_venda_iva` = 0) AS z_20, COUNT_IF(`vl_inserido_externamente_base_venda` IS NULL) AS n_21, CAST(0 AS BIGINT) AS e_21, COUNT_IF(`vl_inserido_externamente_base_venda` = 0) AS z_21, COUNT_IF(`cod_cliente` IS NULL) AS n_22, COUNT_IF(`cod_cliente` IS NOT NULL AND lower(trim(`cod_cliente`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_22, COUNT_IF(`cod_cliente` IS NOT NULL AND trim(`cod_cliente`) RLIKE '^0+([.,]0+)?$') AS z_22, COUNT_IF(`cod_fornecedor` IS NULL) AS n_23, COUNT_IF(`cod_fornecedor` IS NOT NULL AND lower(trim(`cod_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_23, COUNT_IF(`cod_fornecedor` IS NOT NULL AND trim(`cod_fornecedor`) RLIKE '^0+([.,]0+)?$') AS z_23, COUNT_IF(`cod_ordem_cliente` IS NULL) AS n_24, COUNT_IF(`cod_ordem_cliente` IS NOT NULL AND lower(trim(`cod_ordem_cliente`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_24, COUNT_IF(`cod_ordem_cliente` IS NOT NULL AND trim(`cod_ordem_cliente`) RLIKE '^0+([.,]0+)?$') AS z_24, COUNT_IF(`num_item_pedido_venda` IS NULL) AS n_25, COUNT_IF(`num_item_pedido_venda` IS NOT NULL AND lower(trim(`num_item_pedido_venda`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_25, COUNT_IF(`num_item_pedido_venda` IS NOT NULL AND trim(`num_item_pedido_venda`) RLIKE '^0+([.,]0+)?$') AS z_25, COUNT_IF(`num_divisao_remessa` IS NULL) AS n_26, COUNT_IF(`num_divisao_remessa` IS NOT NULL AND lower(trim(`num_divisao_remessa`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_26, COUNT_IF(`num_divisao_remessa` IS NOT NULL AND trim(`num_divisao_remessa`) RLIKE '^0+([.,]0+)?$') AS z_26, COUNT_IF(`ind_material_valor` IS NULL) AS n_27, COUNT_IF(`ind_material_valor` IS NOT NULL AND lower(trim(`ind_material_valor`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_27, COUNT_IF(`ind_material_valor` IS NOT NULL AND trim(`ind_material_valor`) RLIKE '^0+([.,]0+)?$') AS z_27, COUNT_IF(`qt_registrada_antes_contagem` IS NULL) AS n_28, CAST(0 AS BIGINT) AS e_28, COUNT_IF(`qt_registrada_antes_contagem` = 0) AS z_28, COUNT_IF(`qt_quantidade` IS NULL) AS n_29, CAST(0 AS BIGINT) AS e_29, COUNT_IF(`qt_quantidade` = 0) AS z_29, COUNT_IF(`sg_medida_basica_material` IS NULL) AS n_30, COUNT_IF(`sg_medida_basica_material` IS NOT NULL AND lower(trim(`sg_medida_basica_material`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_30, COUNT_IF(`sg_medida_basica_material` IS NOT NULL AND trim(`sg_medida_basica_material`) RLIKE '^0+([.,]0+)?$') AS z_30, COUNT_IF(`vl_contabil` IS NULL) AS n_31, CAST(0 AS BIGINT) AS e_31, COUNT_IF(`vl_contabil` = 0) AS z_31, COUNT_IF(`vl_diferenca` IS NULL) AS n_32, CAST(0 AS BIGINT) AS e_32, COUNT_IF(`vl_diferenca` = 0) AS z_32, COUNT_IF(`cod_moeda` IS NULL) AS n_33, COUNT_IF(`cod_moeda` IS NOT NULL AND lower(trim(`cod_moeda`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_33, COUNT_IF(`cod_moeda` IS NOT NULL AND trim(`cod_moeda`) RLIKE '^0+([.,]0+)?$') AS z_33, COUNT_IF(`dt_lancamento` IS NULL) AS n_34, COUNT_IF(`dt_lancamento` IS NOT NULL AND lower(trim(`dt_lancamento`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_34, COUNT_IF(`dt_lancamento` IS NOT NULL AND trim(`dt_lancamento`) RLIKE '^0+([.,]0+)?$') AS z_34, COUNT_IF(`dt_realizacao_ultima_contagem` IS NULL) AS n_35, COUNT_IF(`dt_realizacao_ultima_contagem` IS NOT NULL AND lower(trim(`dt_realizacao_ultima_contagem`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_35, COUNT_IF(`dt_realizacao_ultima_contagem` IS NOT NULL AND trim(`dt_realizacao_ultima_contagem`) RLIKE '^0+([.,]0+)?$') AS z_35, COUNT_IF(`dt_ultima_modificacao` IS NULL) AS n_36, COUNT_IF(`dt_ultima_modificacao` IS NOT NULL AND lower(trim(`dt_ultima_modificacao`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_36, COUNT_IF(`dt_ultima_modificacao` IS NOT NULL AND trim(`dt_ultima_modificacao`) RLIKE '^0+([.,]0+)?$') AS z_36, COUNT_IF(`cod_usuario_contagem` IS NULL) AS n_37, COUNT_IF(`cod_usuario_contagem` IS NOT NULL AND lower(trim(`cod_usuario_contagem`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_37, COUNT_IF(`cod_usuario_contagem` IS NOT NULL AND trim(`cod_usuario_contagem`) RLIKE '^0+([.,]0+)?$') AS z_37, COUNT_IF(`cod_modificado` IS NULL) AS n_38, COUNT_IF(`cod_modificado` IS NOT NULL AND lower(trim(`cod_modificado`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_38, COUNT_IF(`cod_modificado` IS NOT NULL AND trim(`cod_modificado`) RLIKE '^0+([.,]0+)?$') AS z_38, COUNT_IF(`num_referencia_inventario_fisico` IS NULL) AS n_39, COUNT_IF(`num_referencia_inventario_fisico` IS NOT NULL AND lower(trim(`num_referencia_inventario_fisico`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_39, COUNT_IF(`num_referencia_inventario_fisico` IS NOT NULL AND trim(`num_referencia_inventario_fisico`) RLIKE '^0+([.,]0+)?$') AS z_39, COUNT_IF(`num_material` IS NULL) AS n_40, COUNT_IF(`num_material` IS NOT NULL AND lower(trim(`num_material`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_40, COUNT_IF(`num_material` IS NOT NULL AND trim(`num_material`) RLIKE '^0+([.,]0+)?$') AS z_40, COUNT_IF(`num_ano_material` IS NULL) AS n_41, COUNT_IF(`num_ano_material` IS NOT NULL AND lower(trim(`num_ano_material`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_41, COUNT_IF(`num_ano_material` IS NOT NULL AND trim(`num_ano_material`) RLIKE '^0+([.,]0+)?$') AS z_41, COUNT_IF(`num_item_material` IS NULL) AS n_42, COUNT_IF(`num_item_material` IS NOT NULL AND lower(trim(`num_item_material`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_42, COUNT_IF(`num_item_material` IS NOT NULL AND trim(`num_item_material`) RLIKE '^0+([.,]0+)?$') AS z_42, COUNT_IF(`num_recontagem` IS NULL) AS n_43, COUNT_IF(`num_recontagem` IS NOT NULL AND lower(trim(`num_recontagem`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_43, COUNT_IF(`num_recontagem` IS NOT NULL AND trim(`num_recontagem`) RLIKE '^0+([.,]0+)?$') AS z_43, COUNT_IF(`cod_motivo_diferenca` IS NULL) AS n_44, COUNT_IF(`cod_motivo_diferenca` IS NOT NULL AND lower(trim(`cod_motivo_diferenca`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_44, COUNT_IF(`cod_motivo_diferenca` IS NOT NULL AND trim(`cod_motivo_diferenca`) RLIKE '^0+([.,]0+)?$') AS z_44, COUNT_IF(`dateingest` IS NULL) AS n_45, CAST(0 AS BIGINT) AS e_45, CAST(0 AS BIGINT) AS z_45, COUNT_IF(`yearingest` IS NULL) AS n_46, COUNT_IF(`yearingest` IS NOT NULL AND lower(trim(`yearingest`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_46, COUNT_IF(`yearingest` IS NOT NULL AND trim(`yearingest`) RLIKE '^0+([.,]0+)?$') AS z_46, COUNT_IF(`monthingest` IS NULL) AS n_47, COUNT_IF(`monthingest` IS NOT NULL AND lower(trim(`monthingest`)) IN ('', 'null', 'nan', 'none', '#', '-')) AS e_47, COUNT_IF(`monthingest` IS NOT NULL AND trim(`monthingest`) RLIKE '^0+([.,]0+)?$') AS z_47 FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico),
 perf AS (SELECT stack(48,
  'num_inventario_fisico', 'string', n_0, e_0, z_0,
  'num_item_inventario_fisico', 'string', n_1, e_1, z_1,
  'num_exercicio_fiscal', 'string', n_2, e_2, z_2,
  'cod_material', 'string', n_3, e_3, z_3,
  'desc_material', 'string', n_4, e_4, z_4,
  'tp_categoria_material', 'string', n_5, e_5, z_5,
  'num_lote', 'string', n_6, e_6, z_6,
  'cod_centro', 'string', n_7, e_7, z_7,
  'cod_deposito', 'string', n_8, e_8, z_8,
  'ind_item_contado', 'string', n_9, e_9, z_9,
  'cod_diferenca_item_esta_registrada', 'string', n_10, e_10, z_10,
  'ind_item_recontado', 'string', n_11, e_11, z_11,
  'ind_item_eliminado', 'string', n_12, e_12, z_12,
  'cod_contagem_zero', 'string', n_13, e_13, z_13,
  'tp_estoque_inventario_fisico', 'string', n_14, e_14, z_14,
  'tp_estoque_especial', 'string', n_15, e_15, z_15,
  'tp_inventario_rotativo', 'string', n_16, e_16, z_16,
  'vl_quantidade_contagem_inventario', 'double', n_17, e_17, z_17,
  'vl_diferenca_inventario_entrada_contagem', 'double', n_18, e_18, z_18,
  'vl_precos_venda_excluindo_iva', 'double', n_19, e_19, z_19,
  'vl_precos_venda_iva', 'double', n_20, e_20, z_20,
  'vl_inserido_externamente_base_venda', 'double', n_21, e_21, z_21,
  'cod_cliente', 'string', n_22, e_22, z_22,
  'cod_fornecedor', 'string', n_23, e_23, z_23,
  'cod_ordem_cliente', 'string', n_24, e_24, z_24,
  'num_item_pedido_venda', 'string', n_25, e_25, z_25,
  'num_divisao_remessa', 'string', n_26, e_26, z_26,
  'ind_material_valor', 'string', n_27, e_27, z_27,
  'qt_registrada_antes_contagem', 'decimal(13,3)', n_28, e_28, z_28,
  'qt_quantidade', 'decimal(13,3)', n_29, e_29, z_29,
  'sg_medida_basica_material', 'string', n_30, e_30, z_30,
  'vl_contabil', 'double', n_31, e_31, z_31,
  'vl_diferenca', 'double', n_32, e_32, z_32,
  'cod_moeda', 'string', n_33, e_33, z_33,
  'dt_lancamento', 'string', n_34, e_34, z_34,
  'dt_realizacao_ultima_contagem', 'string', n_35, e_35, z_35,
  'dt_ultima_modificacao', 'string', n_36, e_36, z_36,
  'cod_usuario_contagem', 'string', n_37, e_37, z_37,
  'cod_modificado', 'string', n_38, e_38, z_38,
  'num_referencia_inventario_fisico', 'string', n_39, e_39, z_39,
  'num_material', 'string', n_40, e_40, z_40,
  'num_ano_material', 'string', n_41, e_41, z_41,
  'num_item_material', 'string', n_42, e_42, z_42,
  'num_recontagem', 'string', n_43, e_43, z_43,
  'cod_motivo_diferenca', 'string', n_44, e_44, z_44,
  'dateingest', 'date', n_45, e_45, z_45,
  'yearingest', 'string', n_46, e_46, z_46,
  'monthingest', 'string', n_47, e_47, z_47
 ) AS (coluna, tipo, nulos, vazios, zeros) FROM agg)
SELECT p.coluna, p.tipo, t.total, p.nulos, p.vazios, p.zeros,
 t.total - p.nulos - p.vazios - p.zeros AS uteis_sem_zero,
 CASE WHEN t.total = 0 THEN 'SEM DADOS' WHEN p.nulos = t.total THEN '1. 100% NULO: CONFERIR ORIGEM'
 WHEN p.nulos + p.vazios = t.total THEN '2. SEM PREENCHIMENTO UTIL'
 WHEN p.nulos + p.vazios + p.zeros = t.total THEN '3. APENAS ZERO / AUSENCIA'
 ELSE '4. HA VALORES UTEIS' END AS veredito
FROM perf p CROSS JOIN t ORDER BY veredito, p.coluna;

### 6.1. Colunas “sem fonte”
O CSV não declara nenhuma coluna “sem fonte”. Portanto não há lista para confirmar 100% nula; examine na seção 6 todas as colunas nulas e confira a origem SAP antes de atribuir causa. A referência de inventário é descrita como NULL quando ausente, o que não equivale a “sem fonte”.

In [0]:
SELECT COUNT(*) AS total, COUNT_IF(`num_referencia_inventario_fisico` IS NULL) AS referencia_nula,
 COUNT_IF(`num_referencia_inventario_fisico` IS NOT NULL AND trim(`num_referencia_inventario_fisico`) = '') AS referencia_vazia,
 COUNT_IF(`num_referencia_inventario_fisico` IS NOT NULL AND trim(`num_referencia_inventario_fisico`) <> '') AS referencia_preenchida
FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico;

## 7. Cardinalidade
Uma linha por coluna: constantes, baixa cardinalidade e candidatos a identificador; usuários são apenas contados, não exibidos.

In [0]:
WITH perfil AS (
SELECT 'num_inventario_fisico' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `num_inventario_fisico`) AS distintos_nao_nulos, COUNT_IF(`num_inventario_fisico` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'num_item_inventario_fisico' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `num_item_inventario_fisico`) AS distintos_nao_nulos, COUNT_IF(`num_item_inventario_fisico` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'num_exercicio_fiscal' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `num_exercicio_fiscal`) AS distintos_nao_nulos, COUNT_IF(`num_exercicio_fiscal` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'cod_material' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `cod_material`) AS distintos_nao_nulos, COUNT_IF(`cod_material` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'desc_material' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `desc_material`) AS distintos_nao_nulos, COUNT_IF(`desc_material` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'tp_categoria_material' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `tp_categoria_material`) AS distintos_nao_nulos, COUNT_IF(`tp_categoria_material` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'num_lote' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `num_lote`) AS distintos_nao_nulos, COUNT_IF(`num_lote` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'cod_centro' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `cod_centro`) AS distintos_nao_nulos, COUNT_IF(`cod_centro` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'cod_deposito' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `cod_deposito`) AS distintos_nao_nulos, COUNT_IF(`cod_deposito` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'ind_item_contado' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `ind_item_contado`) AS distintos_nao_nulos, COUNT_IF(`ind_item_contado` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'cod_diferenca_item_esta_registrada' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `cod_diferenca_item_esta_registrada`) AS distintos_nao_nulos, COUNT_IF(`cod_diferenca_item_esta_registrada` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'ind_item_recontado' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `ind_item_recontado`) AS distintos_nao_nulos, COUNT_IF(`ind_item_recontado` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'ind_item_eliminado' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `ind_item_eliminado`) AS distintos_nao_nulos, COUNT_IF(`ind_item_eliminado` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'cod_contagem_zero' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `cod_contagem_zero`) AS distintos_nao_nulos, COUNT_IF(`cod_contagem_zero` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'tp_estoque_inventario_fisico' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `tp_estoque_inventario_fisico`) AS distintos_nao_nulos, COUNT_IF(`tp_estoque_inventario_fisico` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'tp_estoque_especial' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `tp_estoque_especial`) AS distintos_nao_nulos, COUNT_IF(`tp_estoque_especial` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'tp_inventario_rotativo' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `tp_inventario_rotativo`) AS distintos_nao_nulos, COUNT_IF(`tp_inventario_rotativo` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'vl_quantidade_contagem_inventario' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `vl_quantidade_contagem_inventario`) AS distintos_nao_nulos, COUNT_IF(`vl_quantidade_contagem_inventario` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'vl_diferenca_inventario_entrada_contagem' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `vl_diferenca_inventario_entrada_contagem`) AS distintos_nao_nulos, COUNT_IF(`vl_diferenca_inventario_entrada_contagem` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'vl_precos_venda_excluindo_iva' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `vl_precos_venda_excluindo_iva`) AS distintos_nao_nulos, COUNT_IF(`vl_precos_venda_excluindo_iva` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'vl_precos_venda_iva' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `vl_precos_venda_iva`) AS distintos_nao_nulos, COUNT_IF(`vl_precos_venda_iva` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'vl_inserido_externamente_base_venda' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `vl_inserido_externamente_base_venda`) AS distintos_nao_nulos, COUNT_IF(`vl_inserido_externamente_base_venda` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'cod_cliente' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `cod_cliente`) AS distintos_nao_nulos, COUNT_IF(`cod_cliente` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'cod_fornecedor' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `cod_fornecedor`) AS distintos_nao_nulos, COUNT_IF(`cod_fornecedor` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'cod_ordem_cliente' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `cod_ordem_cliente`) AS distintos_nao_nulos, COUNT_IF(`cod_ordem_cliente` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'num_item_pedido_venda' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `num_item_pedido_venda`) AS distintos_nao_nulos, COUNT_IF(`num_item_pedido_venda` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'num_divisao_remessa' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `num_divisao_remessa`) AS distintos_nao_nulos, COUNT_IF(`num_divisao_remessa` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'ind_material_valor' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `ind_material_valor`) AS distintos_nao_nulos, COUNT_IF(`ind_material_valor` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'qt_registrada_antes_contagem' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `qt_registrada_antes_contagem`) AS distintos_nao_nulos, COUNT_IF(`qt_registrada_antes_contagem` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'qt_quantidade' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `qt_quantidade`) AS distintos_nao_nulos, COUNT_IF(`qt_quantidade` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'sg_medida_basica_material' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `sg_medida_basica_material`) AS distintos_nao_nulos, COUNT_IF(`sg_medida_basica_material` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'vl_contabil' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `vl_contabil`) AS distintos_nao_nulos, COUNT_IF(`vl_contabil` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'vl_diferenca' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `vl_diferenca`) AS distintos_nao_nulos, COUNT_IF(`vl_diferenca` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'cod_moeda' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `cod_moeda`) AS distintos_nao_nulos, COUNT_IF(`cod_moeda` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'dt_lancamento' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `dt_lancamento`) AS distintos_nao_nulos, COUNT_IF(`dt_lancamento` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'dt_realizacao_ultima_contagem' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `dt_realizacao_ultima_contagem`) AS distintos_nao_nulos, COUNT_IF(`dt_realizacao_ultima_contagem` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'dt_ultima_modificacao' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `dt_ultima_modificacao`) AS distintos_nao_nulos, COUNT_IF(`dt_ultima_modificacao` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'cod_usuario_contagem' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `cod_usuario_contagem`) AS distintos_nao_nulos, COUNT_IF(`cod_usuario_contagem` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'cod_modificado' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `cod_modificado`) AS distintos_nao_nulos, COUNT_IF(`cod_modificado` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'num_referencia_inventario_fisico' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `num_referencia_inventario_fisico`) AS distintos_nao_nulos, COUNT_IF(`num_referencia_inventario_fisico` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'num_material' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `num_material`) AS distintos_nao_nulos, COUNT_IF(`num_material` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'num_ano_material' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `num_ano_material`) AS distintos_nao_nulos, COUNT_IF(`num_ano_material` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'num_item_material' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `num_item_material`) AS distintos_nao_nulos, COUNT_IF(`num_item_material` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'num_recontagem' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `num_recontagem`) AS distintos_nao_nulos, COUNT_IF(`num_recontagem` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'cod_motivo_diferenca' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `cod_motivo_diferenca`) AS distintos_nao_nulos, COUNT_IF(`cod_motivo_diferenca` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'dateingest' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `dateingest`) AS distintos_nao_nulos, COUNT_IF(`dateingest` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'yearingest' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `yearingest`) AS distintos_nao_nulos, COUNT_IF(`yearingest` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'monthingest' AS coluna, COUNT(*) AS linhas, COUNT(DISTINCT `monthingest`) AS distintos_nao_nulos, COUNT_IF(`monthingest` IS NULL) AS nulos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
) SELECT coluna, linhas, distintos_nao_nulos, nulos, CASE WHEN distintos_nao_nulos <= 1 THEN 'CONSTANTE/VAZIA' WHEN distintos_nao_nulos <= 8 THEN 'BAIXA CARDINALIDADE' WHEN distintos_nao_nulos >= linhas * 0.9 THEN 'CANDIDATA A IDENTIFICADOR' ELSE 'INTERMEDIARIA' END AS classe FROM perfil ORDER BY distintos_nao_nulos, coluna;

## 8. Domínio
Top 8 das dimensões categóricas, inclusive ausências; não mostre valores de usuário.

In [0]:
WITH dominios AS (
SELECT 'cod_centro' AS coluna, COALESCE(CAST(`cod_centro` AS STRING),'<NULL>') AS valor, COUNT(*) AS linhas, ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC, COALESCE(CAST(`cod_centro` AS STRING),'<NULL>')) AS posicao FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico GROUP BY `cod_centro`
UNION ALL
SELECT 'cod_deposito' AS coluna, COALESCE(CAST(`cod_deposito` AS STRING),'<NULL>') AS valor, COUNT(*) AS linhas, ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC, COALESCE(CAST(`cod_deposito` AS STRING),'<NULL>')) AS posicao FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico GROUP BY `cod_deposito`
UNION ALL
SELECT 'tp_categoria_material' AS coluna, COALESCE(CAST(`tp_categoria_material` AS STRING),'<NULL>') AS valor, COUNT(*) AS linhas, ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC, COALESCE(CAST(`tp_categoria_material` AS STRING),'<NULL>')) AS posicao FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico GROUP BY `tp_categoria_material`
UNION ALL
SELECT 'tp_estoque_inventario_fisico' AS coluna, COALESCE(CAST(`tp_estoque_inventario_fisico` AS STRING),'<NULL>') AS valor, COUNT(*) AS linhas, ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC, COALESCE(CAST(`tp_estoque_inventario_fisico` AS STRING),'<NULL>')) AS posicao FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico GROUP BY `tp_estoque_inventario_fisico`
UNION ALL
SELECT 'tp_estoque_especial' AS coluna, COALESCE(CAST(`tp_estoque_especial` AS STRING),'<NULL>') AS valor, COUNT(*) AS linhas, ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC, COALESCE(CAST(`tp_estoque_especial` AS STRING),'<NULL>')) AS posicao FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico GROUP BY `tp_estoque_especial`
UNION ALL
SELECT 'tp_inventario_rotativo' AS coluna, COALESCE(CAST(`tp_inventario_rotativo` AS STRING),'<NULL>') AS valor, COUNT(*) AS linhas, ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC, COALESCE(CAST(`tp_inventario_rotativo` AS STRING),'<NULL>')) AS posicao FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico GROUP BY `tp_inventario_rotativo`
UNION ALL
SELECT 'cod_moeda' AS coluna, COALESCE(CAST(`cod_moeda` AS STRING),'<NULL>') AS valor, COUNT(*) AS linhas, ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC, COALESCE(CAST(`cod_moeda` AS STRING),'<NULL>')) AS posicao FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico GROUP BY `cod_moeda`
UNION ALL
SELECT 'sg_medida_basica_material' AS coluna, COALESCE(CAST(`sg_medida_basica_material` AS STRING),'<NULL>') AS valor, COUNT(*) AS linhas, ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC, COALESCE(CAST(`sg_medida_basica_material` AS STRING),'<NULL>')) AS posicao FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico GROUP BY `sg_medida_basica_material`
UNION ALL
SELECT 'cod_motivo_diferenca' AS coluna, COALESCE(CAST(`cod_motivo_diferenca` AS STRING),'<NULL>') AS valor, COUNT(*) AS linhas, ROW_NUMBER() OVER (ORDER BY COUNT(*) DESC, COALESCE(CAST(`cod_motivo_diferenca` AS STRING),'<NULL>')) AS posicao FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico GROUP BY `cod_motivo_diferenca`
) SELECT coluna, valor, linhas FROM dominios WHERE posicao <= 8 ORDER BY coluna, linhas DESC;

## 9. Indicadores
Confira X, vazio e valores inesperados, sem converter silenciosamente em boolean.

In [0]:
WITH f AS (
SELECT 'ind_item_contado' AS indicador, CASE WHEN `ind_item_contado` IS NULL THEN '<NULL>' WHEN trim(`ind_item_contado`) = '' THEN '<VAZIO>' ELSE trim(`ind_item_contado`) END AS valor, COUNT(*) AS linhas FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico GROUP BY `ind_item_contado`
UNION ALL
SELECT 'cod_diferenca_item_esta_registrada' AS indicador, CASE WHEN `cod_diferenca_item_esta_registrada` IS NULL THEN '<NULL>' WHEN trim(`cod_diferenca_item_esta_registrada`) = '' THEN '<VAZIO>' ELSE trim(`cod_diferenca_item_esta_registrada`) END AS valor, COUNT(*) AS linhas FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico GROUP BY `cod_diferenca_item_esta_registrada`
UNION ALL
SELECT 'ind_item_recontado' AS indicador, CASE WHEN `ind_item_recontado` IS NULL THEN '<NULL>' WHEN trim(`ind_item_recontado`) = '' THEN '<VAZIO>' ELSE trim(`ind_item_recontado`) END AS valor, COUNT(*) AS linhas FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico GROUP BY `ind_item_recontado`
UNION ALL
SELECT 'ind_item_eliminado' AS indicador, CASE WHEN `ind_item_eliminado` IS NULL THEN '<NULL>' WHEN trim(`ind_item_eliminado`) = '' THEN '<VAZIO>' ELSE trim(`ind_item_eliminado`) END AS valor, COUNT(*) AS linhas FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico GROUP BY `ind_item_eliminado`
UNION ALL
SELECT 'cod_contagem_zero' AS indicador, CASE WHEN `cod_contagem_zero` IS NULL THEN '<NULL>' WHEN trim(`cod_contagem_zero`) = '' THEN '<VAZIO>' ELSE trim(`cod_contagem_zero`) END AS valor, COUNT(*) AS linhas FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico GROUP BY `cod_contagem_zero`
UNION ALL
SELECT 'ind_material_valor' AS indicador, CASE WHEN `ind_material_valor` IS NULL THEN '<NULL>' WHEN trim(`ind_material_valor`) = '' THEN '<VAZIO>' ELSE trim(`ind_material_valor`) END AS valor, COUNT(*) AS linhas FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico GROUP BY `ind_material_valor`
) SELECT indicador, valor, SUM(linhas) AS linhas FROM f GROUP BY indicador, valor ORDER BY indicador, linhas DESC;

## 10. Perfil numérico
Extremos, média, mediana, negativos e zeros por campo. Mediana via percentil aproximado; não misture quantidade com moeda.

In [0]:
SELECT 'vl_quantidade_contagem_inventario' AS coluna, COUNT(`vl_quantidade_contagem_inventario`) AS preenchidos, MIN(`vl_quantidade_contagem_inventario`) AS minimo, MAX(`vl_quantidade_contagem_inventario`) AS maximo, AVG(`vl_quantidade_contagem_inventario`) AS media, percentile_approx(`vl_quantidade_contagem_inventario`, 0.5) AS mediana_aproximada, COUNT_IF(`vl_quantidade_contagem_inventario` < 0) AS negativos, COUNT_IF(`vl_quantidade_contagem_inventario` = 0) AS zeros FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'vl_diferenca_inventario_entrada_contagem' AS coluna, COUNT(`vl_diferenca_inventario_entrada_contagem`) AS preenchidos, MIN(`vl_diferenca_inventario_entrada_contagem`) AS minimo, MAX(`vl_diferenca_inventario_entrada_contagem`) AS maximo, AVG(`vl_diferenca_inventario_entrada_contagem`) AS media, percentile_approx(`vl_diferenca_inventario_entrada_contagem`, 0.5) AS mediana_aproximada, COUNT_IF(`vl_diferenca_inventario_entrada_contagem` < 0) AS negativos, COUNT_IF(`vl_diferenca_inventario_entrada_contagem` = 0) AS zeros FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'vl_precos_venda_excluindo_iva' AS coluna, COUNT(`vl_precos_venda_excluindo_iva`) AS preenchidos, MIN(`vl_precos_venda_excluindo_iva`) AS minimo, MAX(`vl_precos_venda_excluindo_iva`) AS maximo, AVG(`vl_precos_venda_excluindo_iva`) AS media, percentile_approx(`vl_precos_venda_excluindo_iva`, 0.5) AS mediana_aproximada, COUNT_IF(`vl_precos_venda_excluindo_iva` < 0) AS negativos, COUNT_IF(`vl_precos_venda_excluindo_iva` = 0) AS zeros FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'vl_precos_venda_iva' AS coluna, COUNT(`vl_precos_venda_iva`) AS preenchidos, MIN(`vl_precos_venda_iva`) AS minimo, MAX(`vl_precos_venda_iva`) AS maximo, AVG(`vl_precos_venda_iva`) AS media, percentile_approx(`vl_precos_venda_iva`, 0.5) AS mediana_aproximada, COUNT_IF(`vl_precos_venda_iva` < 0) AS negativos, COUNT_IF(`vl_precos_venda_iva` = 0) AS zeros FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'vl_inserido_externamente_base_venda' AS coluna, COUNT(`vl_inserido_externamente_base_venda`) AS preenchidos, MIN(`vl_inserido_externamente_base_venda`) AS minimo, MAX(`vl_inserido_externamente_base_venda`) AS maximo, AVG(`vl_inserido_externamente_base_venda`) AS media, percentile_approx(`vl_inserido_externamente_base_venda`, 0.5) AS mediana_aproximada, COUNT_IF(`vl_inserido_externamente_base_venda` < 0) AS negativos, COUNT_IF(`vl_inserido_externamente_base_venda` = 0) AS zeros FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'qt_registrada_antes_contagem' AS coluna, COUNT(`qt_registrada_antes_contagem`) AS preenchidos, MIN(`qt_registrada_antes_contagem`) AS minimo, MAX(`qt_registrada_antes_contagem`) AS maximo, AVG(`qt_registrada_antes_contagem`) AS media, percentile_approx(`qt_registrada_antes_contagem`, 0.5) AS mediana_aproximada, COUNT_IF(`qt_registrada_antes_contagem` < 0) AS negativos, COUNT_IF(`qt_registrada_antes_contagem` = 0) AS zeros FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'qt_quantidade' AS coluna, COUNT(`qt_quantidade`) AS preenchidos, MIN(`qt_quantidade`) AS minimo, MAX(`qt_quantidade`) AS maximo, AVG(`qt_quantidade`) AS media, percentile_approx(`qt_quantidade`, 0.5) AS mediana_aproximada, COUNT_IF(`qt_quantidade` < 0) AS negativos, COUNT_IF(`qt_quantidade` = 0) AS zeros FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'vl_contabil' AS coluna, COUNT(`vl_contabil`) AS preenchidos, MIN(`vl_contabil`) AS minimo, MAX(`vl_contabil`) AS maximo, AVG(`vl_contabil`) AS media, percentile_approx(`vl_contabil`, 0.5) AS mediana_aproximada, COUNT_IF(`vl_contabil` < 0) AS negativos, COUNT_IF(`vl_contabil` = 0) AS zeros FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'vl_diferenca' AS coluna, COUNT(`vl_diferenca`) AS preenchidos, MIN(`vl_diferenca`) AS minimo, MAX(`vl_diferenca`) AS maximo, AVG(`vl_diferenca`) AS media, percentile_approx(`vl_diferenca`, 0.5) AS mediana_aproximada, COUNT_IF(`vl_diferenca` < 0) AS negativos, COUNT_IF(`vl_diferenca` = 0) AS zeros FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico;

## 11. Totais para conciliação
Somas independentes para cada campo numérico, inclusive quantidades. Compare `double` com tolerância de 0,005 por item, e registre moeda/unidade.

In [0]:
SELECT `cod_centro`, `cod_moeda`, `sg_medida_basica_material`, COUNT(*) AS linhas,
 SUM(`vl_quantidade_contagem_inventario`) AS `vl_quantidade_contagem_inventario`, SUM(`vl_diferenca_inventario_entrada_contagem`) AS `vl_diferenca_inventario_entrada_contagem`, SUM(`vl_precos_venda_excluindo_iva`) AS `vl_precos_venda_excluindo_iva`, SUM(`vl_precos_venda_iva`) AS `vl_precos_venda_iva`, SUM(`vl_inserido_externamente_base_venda`) AS `vl_inserido_externamente_base_venda`, SUM(`qt_registrada_antes_contagem`) AS `qt_registrada_antes_contagem`, SUM(`qt_quantidade`) AS `qt_quantidade`, SUM(`vl_contabil`) AS `vl_contabil`, SUM(`vl_diferenca`) AS `vl_diferenca`
FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico GROUP BY `cod_centro`, `cod_moeda`, `sg_medida_basica_material` ORDER BY linhas DESC;

## 12. Datas
Meça padrões e sentinelas; parse é teste, não correção automática. `dateingest` é DATE.

In [0]:
SELECT 'dt_lancamento' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_lancamento` IS NULL OR trim(`dt_lancamento`) = '') AS ausentes, COUNT_IF(trim(`dt_lancamento`) = '00000000') AS sentinela_zero, COUNT_IF(trim(`dt_lancamento`) LIKE '9999-12-31%') AS sentinela_futura, COUNT_IF(trim(`dt_lancamento`) RLIKE '^[0-9]{8}$') AS formato_aaaammdd, COUNT_IF(trim(`dt_lancamento`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}') AS formato_iso, COUNT_IF(trim(`dt_lancamento`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}$') AS formato_br, MIN(try_to_timestamp(`dt_lancamento`)) AS menor_parse_padrao, MAX(try_to_timestamp(`dt_lancamento`)) AS maior_parse_padrao FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'dt_realizacao_ultima_contagem' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_realizacao_ultima_contagem` IS NULL OR trim(`dt_realizacao_ultima_contagem`) = '') AS ausentes, COUNT_IF(trim(`dt_realizacao_ultima_contagem`) = '00000000') AS sentinela_zero, COUNT_IF(trim(`dt_realizacao_ultima_contagem`) LIKE '9999-12-31%') AS sentinela_futura, COUNT_IF(trim(`dt_realizacao_ultima_contagem`) RLIKE '^[0-9]{8}$') AS formato_aaaammdd, COUNT_IF(trim(`dt_realizacao_ultima_contagem`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}') AS formato_iso, COUNT_IF(trim(`dt_realizacao_ultima_contagem`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}$') AS formato_br, MIN(try_to_timestamp(`dt_realizacao_ultima_contagem`)) AS menor_parse_padrao, MAX(try_to_timestamp(`dt_realizacao_ultima_contagem`)) AS maior_parse_padrao FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'dt_ultima_modificacao' AS coluna, COUNT(*) AS linhas, COUNT_IF(`dt_ultima_modificacao` IS NULL OR trim(`dt_ultima_modificacao`) = '') AS ausentes, COUNT_IF(trim(`dt_ultima_modificacao`) = '00000000') AS sentinela_zero, COUNT_IF(trim(`dt_ultima_modificacao`) LIKE '9999-12-31%') AS sentinela_futura, COUNT_IF(trim(`dt_ultima_modificacao`) RLIKE '^[0-9]{8}$') AS formato_aaaammdd, COUNT_IF(trim(`dt_ultima_modificacao`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}') AS formato_iso, COUNT_IF(trim(`dt_ultima_modificacao`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}$') AS formato_br, MIN(try_to_timestamp(`dt_ultima_modificacao`)) AS menor_parse_padrao, MAX(try_to_timestamp(`dt_ultima_modificacao`)) AS maior_parse_padrao FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico;

In [0]:
SELECT MIN(`dateingest`) AS primeira_ingestao, MAX(`dateingest`) AS ultima_ingestao,
 COUNT_IF(`dateingest` IS NULL) AS ingestao_nula,
 MIN(to_date(try_to_timestamp(CASE WHEN `dt_lancamento` RLIKE '^[0-9]{8}$' AND `dt_lancamento` <> '00000000' THEN `dt_lancamento` END,'yyyyMMdd'))) AS menor_lancamento_aaaammdd,
 MAX(to_date(try_to_timestamp(CASE WHEN `dt_lancamento` RLIKE '^[0-9]{8}$' AND `dt_lancamento` <> '00000000' THEN `dt_lancamento` END,'yyyyMMdd'))) AS maior_lancamento_aaaammdd
FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico;

## 13. Códigos
Meça zeros à esquerda e colisões ao removê-los, sempre preservando os valores originais no Datalake.

In [0]:
SELECT 'cod_material' AS coluna, COUNT(`cod_material`) AS preenchidos, MIN(length(`cod_material`)) AS menor_tamanho, MAX(length(`cod_material`)) AS maior_tamanho, COUNT_IF(`cod_material` RLIKE '^0+[0-9]+$') AS com_zero_a_esquerda, COUNT(DISTINCT `cod_material`) AS distintos_originais, COUNT(DISTINCT CASE WHEN `cod_material` RLIKE '^[0-9]+$' THEN COALESCE(NULLIF(regexp_replace(`cod_material`, '^0+', ''), ''), '0') ELSE `cod_material` END) AS distintos_normalizados FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'num_inventario_fisico' AS coluna, COUNT(`num_inventario_fisico`) AS preenchidos, MIN(length(`num_inventario_fisico`)) AS menor_tamanho, MAX(length(`num_inventario_fisico`)) AS maior_tamanho, COUNT_IF(`num_inventario_fisico` RLIKE '^0+[0-9]+$') AS com_zero_a_esquerda, COUNT(DISTINCT `num_inventario_fisico`) AS distintos_originais, COUNT(DISTINCT CASE WHEN `num_inventario_fisico` RLIKE '^[0-9]+$' THEN COALESCE(NULLIF(regexp_replace(`num_inventario_fisico`, '^0+', ''), ''), '0') ELSE `num_inventario_fisico` END) AS distintos_normalizados FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'num_item_inventario_fisico' AS coluna, COUNT(`num_item_inventario_fisico`) AS preenchidos, MIN(length(`num_item_inventario_fisico`)) AS menor_tamanho, MAX(length(`num_item_inventario_fisico`)) AS maior_tamanho, COUNT_IF(`num_item_inventario_fisico` RLIKE '^0+[0-9]+$') AS com_zero_a_esquerda, COUNT(DISTINCT `num_item_inventario_fisico`) AS distintos_originais, COUNT(DISTINCT CASE WHEN `num_item_inventario_fisico` RLIKE '^[0-9]+$' THEN COALESCE(NULLIF(regexp_replace(`num_item_inventario_fisico`, '^0+', ''), ''), '0') ELSE `num_item_inventario_fisico` END) AS distintos_normalizados FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'cod_centro' AS coluna, COUNT(`cod_centro`) AS preenchidos, MIN(length(`cod_centro`)) AS menor_tamanho, MAX(length(`cod_centro`)) AS maior_tamanho, COUNT_IF(`cod_centro` RLIKE '^0+[0-9]+$') AS com_zero_a_esquerda, COUNT(DISTINCT `cod_centro`) AS distintos_originais, COUNT(DISTINCT CASE WHEN `cod_centro` RLIKE '^[0-9]+$' THEN COALESCE(NULLIF(regexp_replace(`cod_centro`, '^0+', ''), ''), '0') ELSE `cod_centro` END) AS distintos_normalizados FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'cod_deposito' AS coluna, COUNT(`cod_deposito`) AS preenchidos, MIN(length(`cod_deposito`)) AS menor_tamanho, MAX(length(`cod_deposito`)) AS maior_tamanho, COUNT_IF(`cod_deposito` RLIKE '^0+[0-9]+$') AS com_zero_a_esquerda, COUNT(DISTINCT `cod_deposito`) AS distintos_originais, COUNT(DISTINCT CASE WHEN `cod_deposito` RLIKE '^[0-9]+$' THEN COALESCE(NULLIF(regexp_replace(`cod_deposito`, '^0+', ''), ''), '0') ELSE `cod_deposito` END) AS distintos_normalizados FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'cod_cliente' AS coluna, COUNT(`cod_cliente`) AS preenchidos, MIN(length(`cod_cliente`)) AS menor_tamanho, MAX(length(`cod_cliente`)) AS maior_tamanho, COUNT_IF(`cod_cliente` RLIKE '^0+[0-9]+$') AS com_zero_a_esquerda, COUNT(DISTINCT `cod_cliente`) AS distintos_originais, COUNT(DISTINCT CASE WHEN `cod_cliente` RLIKE '^[0-9]+$' THEN COALESCE(NULLIF(regexp_replace(`cod_cliente`, '^0+', ''), ''), '0') ELSE `cod_cliente` END) AS distintos_normalizados FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'cod_fornecedor' AS coluna, COUNT(`cod_fornecedor`) AS preenchidos, MIN(length(`cod_fornecedor`)) AS menor_tamanho, MAX(length(`cod_fornecedor`)) AS maior_tamanho, COUNT_IF(`cod_fornecedor` RLIKE '^0+[0-9]+$') AS com_zero_a_esquerda, COUNT(DISTINCT `cod_fornecedor`) AS distintos_originais, COUNT(DISTINCT CASE WHEN `cod_fornecedor` RLIKE '^[0-9]+$' THEN COALESCE(NULLIF(regexp_replace(`cod_fornecedor`, '^0+', ''), ''), '0') ELSE `cod_fornecedor` END) AS distintos_normalizados FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'num_material' AS coluna, COUNT(`num_material`) AS preenchidos, MIN(length(`num_material`)) AS menor_tamanho, MAX(length(`num_material`)) AS maior_tamanho, COUNT_IF(`num_material` RLIKE '^0+[0-9]+$') AS com_zero_a_esquerda, COUNT(DISTINCT `num_material`) AS distintos_originais, COUNT(DISTINCT CASE WHEN `num_material` RLIKE '^[0-9]+$' THEN COALESCE(NULLIF(regexp_replace(`num_material`, '^0+', ''), ''), '0') ELSE `num_material` END) AS distintos_normalizados FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
UNION ALL
SELECT 'num_recontagem' AS coluna, COUNT(`num_recontagem`) AS preenchidos, MIN(length(`num_recontagem`)) AS menor_tamanho, MAX(length(`num_recontagem`)) AS maior_tamanho, COUNT_IF(`num_recontagem` RLIKE '^0+[0-9]+$') AS com_zero_a_esquerda, COUNT(DISTINCT `num_recontagem`) AS distintos_originais, COUNT(DISTINCT CASE WHEN `num_recontagem` RLIKE '^[0-9]+$' THEN COALESCE(NULLIF(regexp_replace(`num_recontagem`, '^0+', ''), ''), '0') ELSE `num_recontagem` END) AS distintos_normalizados FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico;

## 14. Checksum
Hash de todas as 48 colunas para separar linhas 100% idênticas; nenhum dado pessoal aparece no resultado.

In [0]:
WITH h AS (SELECT md5(to_json(named_struct('num_inventario_fisico', `num_inventario_fisico`, 'num_item_inventario_fisico', `num_item_inventario_fisico`, 'num_exercicio_fiscal', `num_exercicio_fiscal`, 'cod_material', `cod_material`, 'desc_material', `desc_material`, 'tp_categoria_material', `tp_categoria_material`, 'num_lote', `num_lote`, 'cod_centro', `cod_centro`, 'cod_deposito', `cod_deposito`, 'ind_item_contado', `ind_item_contado`, 'cod_diferenca_item_esta_registrada', `cod_diferenca_item_esta_registrada`, 'ind_item_recontado', `ind_item_recontado`, 'ind_item_eliminado', `ind_item_eliminado`, 'cod_contagem_zero', `cod_contagem_zero`, 'tp_estoque_inventario_fisico', `tp_estoque_inventario_fisico`, 'tp_estoque_especial', `tp_estoque_especial`, 'tp_inventario_rotativo', `tp_inventario_rotativo`, 'vl_quantidade_contagem_inventario', `vl_quantidade_contagem_inventario`, 'vl_diferenca_inventario_entrada_contagem', `vl_diferenca_inventario_entrada_contagem`, 'vl_precos_venda_excluindo_iva', `vl_precos_venda_excluindo_iva`, 'vl_precos_venda_iva', `vl_precos_venda_iva`, 'vl_inserido_externamente_base_venda', `vl_inserido_externamente_base_venda`, 'cod_cliente', `cod_cliente`, 'cod_fornecedor', `cod_fornecedor`, 'cod_ordem_cliente', `cod_ordem_cliente`, 'num_item_pedido_venda', `num_item_pedido_venda`, 'num_divisao_remessa', `num_divisao_remessa`, 'ind_material_valor', `ind_material_valor`, 'qt_registrada_antes_contagem', `qt_registrada_antes_contagem`, 'qt_quantidade', `qt_quantidade`, 'sg_medida_basica_material', `sg_medida_basica_material`, 'vl_contabil', `vl_contabil`, 'vl_diferenca', `vl_diferenca`, 'cod_moeda', `cod_moeda`, 'dt_lancamento', `dt_lancamento`, 'dt_realizacao_ultima_contagem', `dt_realizacao_ultima_contagem`, 'dt_ultima_modificacao', `dt_ultima_modificacao`, 'cod_usuario_contagem', `cod_usuario_contagem`, 'cod_modificado', `cod_modificado`, 'num_referencia_inventario_fisico', `num_referencia_inventario_fisico`, 'num_material', `num_material`, 'num_ano_material', `num_ano_material`, 'num_item_material', `num_item_material`, 'num_recontagem', `num_recontagem`, 'cod_motivo_diferenca', `cod_motivo_diferenca`, 'dateingest', `dateingest`, 'yearingest', `yearingest`, 'monthingest', `monthingest`))) AS hash_linha FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico), g AS (SELECT hash_linha, COUNT(*) AS linhas FROM h GROUP BY hash_linha)
SELECT SUM(linhas) AS linhas, COUNT(*) AS hashes_distintos, COALESCE(SUM(linhas - 1),0) AS repeticoes_exatas, MAX(linhas) AS maior_grupo_identico FROM g;

## 15. Amostra de 20 linhas
Inspeção de registros comparáveis; usuários mascarados, sem `SELECT *`.

In [0]:
SELECT `num_inventario_fisico`, `num_item_inventario_fisico`, `num_exercicio_fiscal`, `cod_material`, `desc_material`, `tp_categoria_material`, `num_lote`, `cod_centro`, `cod_deposito`, `ind_item_contado`, `cod_diferenca_item_esta_registrada`, `ind_item_recontado`, `ind_item_eliminado`, `cod_contagem_zero`, `tp_estoque_inventario_fisico`, `tp_estoque_especial`, `tp_inventario_rotativo`, `vl_quantidade_contagem_inventario`, `vl_diferenca_inventario_entrada_contagem`, `vl_precos_venda_excluindo_iva`, `vl_precos_venda_iva`, `vl_inserido_externamente_base_venda`, `cod_cliente`, `cod_fornecedor`, `cod_ordem_cliente`, `num_item_pedido_venda`, `num_divisao_remessa`, `ind_material_valor`, `qt_registrada_antes_contagem`, `qt_quantidade`, `sg_medida_basica_material`, `vl_contabil`, `vl_diferenca`, `cod_moeda`, `dt_lancamento`, `dt_realizacao_ultima_contagem`, `dt_ultima_modificacao`, substring(md5(CAST(`cod_usuario_contagem` AS STRING)),1,8) AS `cod_usuario_contagem_hash`, substring(md5(CAST(`cod_modificado` AS STRING)),1,8) AS `cod_modificado_hash`, `num_referencia_inventario_fisico`, `num_material`, `num_ano_material`, `num_item_material`, `num_recontagem`, `cod_motivo_diferenca`, `dateingest`, `yearingest`, `monthingest` FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico ORDER BY `cod_centro`, `num_exercicio_fiscal`, `num_inventario_fisico`, `num_item_inventario_fisico` LIMIT 20;

## 16. Distribuição interna
Cruzamento por centro, exercício, depósito e tipo de estoque para identificar recortes comparáveis com a tela SAP.

In [0]:
SELECT `cod_centro`, `num_exercicio_fiscal`, `cod_deposito`, `tp_estoque_inventario_fisico`, `tp_estoque_especial`, COUNT(*) AS linhas,
 COUNT(DISTINCT `num_inventario_fisico`) AS documentos FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico
GROUP BY `cod_centro`, `num_exercicio_fiscal`, `cod_deposito`, `tp_estoque_inventario_fisico`, `tp_estoque_especial` ORDER BY linhas DESC LIMIT 100;

## 17. Freshness e histórico
`dateingest`, `yearingest` e `monthingest` existem. Várias datas não provam histórico: verifique repetição da chave entre cargas.

In [0]:
SELECT `dateingest`, `yearingest`, `monthingest`, COUNT(*) AS linhas, COUNT(DISTINCT `num_inventario_fisico`) AS documentos
FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico GROUP BY `dateingest`, `yearingest`, `monthingest` ORDER BY `dateingest` DESC LIMIT 100;

In [0]:
WITH k AS (SELECT `cod_centro`, `num_exercicio_fiscal`, `num_inventario_fisico`, `num_item_inventario_fisico`, COUNT(DISTINCT `dateingest`) AS datas_ingestao, COUNT(*) AS linhas FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico GROUP BY `cod_centro`, `num_exercicio_fiscal`, `num_inventario_fisico`, `num_item_inventario_fisico`)
SELECT COUNT(*) AS chaves_candidatas, COUNT_IF(datas_ingestao > 1) AS chaves_em_multiplas_datas,
 SUM(CASE WHEN datas_ingestao > 1 THEN linhas ELSE 0 END) AS linhas_em_multiplas_datas FROM k;

## 18. Testes específicos da MI24
São hipóteses testáveis: contagem/recontagem, diferenças, datas, documentos de material e de-para da unidade. Não conclua divergência sem comparar com SAP e os filtros da tela.

In [0]:
SELECT `cod_centro`, `ind_item_contado`, `ind_item_recontado`, `ind_item_eliminado`, `cod_contagem_zero`, `cod_diferenca_item_esta_registrada`, COUNT(*) AS linhas,
 COUNT_IF(`qt_quantidade` = 0) AS quantidade_zero, COUNT_IF(`qt_quantidade` < 0) AS quantidade_negativa,
 COUNT_IF(`vl_diferenca` <> 0) AS diferenca_nao_zero
FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico GROUP BY `cod_centro`, `ind_item_contado`, `ind_item_recontado`, `ind_item_eliminado`, `cod_contagem_zero`, `cod_diferenca_item_esta_registrada` ORDER BY linhas DESC LIMIT 100;

In [0]:
SELECT COUNT(*) AS linhas,
 COUNT_IF(`ind_item_contado` = 'X' AND `qt_quantidade` IS NULL) AS contado_sem_quantidade,
 COUNT_IF(`cod_contagem_zero` = 'X' AND `qt_quantidade` <> 0) AS contagem_zero_com_quantidade_nao_zero,
 COUNT_IF(`ind_item_recontado` = 'X' AND (`num_recontagem` IS NULL OR trim(`num_recontagem`) = '')) AS recontado_sem_documento,
 COUNT_IF(`cod_diferenca_item_esta_registrada` = 'X' AND `vl_diferenca` IS NULL) AS diferenca_registrada_sem_valor,
 COUNT_IF(`num_material` IS NOT NULL AND trim(`num_material`) <> '' AND (`num_ano_material` IS NULL OR trim(`num_ano_material`) = '')) AS documento_material_sem_ano,
 COUNT_IF(`num_material` IS NOT NULL AND trim(`num_material`) <> '' AND (`num_item_material` IS NULL OR trim(`num_item_material`) = '')) AS documento_material_sem_item,
 COUNT_IF(`qt_quantidade` IS NOT NULL AND `qt_registrada_antes_contagem` IS NOT NULL AND `qt_quantidade` <> `qt_registrada_antes_contagem` AND (`vl_diferenca` IS NULL OR `vl_diferenca` = 0)) AS quantidade_diferente_sem_valor_diferenca
FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico;

In [0]:
SELECT `cod_centro`, `sg_medida_basica_material`, COUNT(*) AS linhas,
 COUNT(DISTINCT `cod_material`) AS materiais, COUNT_IF(`sg_medida_basica_material` IS NULL OR trim(`sg_medida_basica_material`) = '') AS unidade_ausente
FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico GROUP BY `cod_centro`, `sg_medida_basica_material` ORDER BY linhas DESC LIMIT 100;

In [0]:
SELECT COUNT(*) AS linhas,
 COUNT_IF(`dt_lancamento` RLIKE '^[0-9]{8}$' AND `dt_realizacao_ultima_contagem` RLIKE '^[0-9]{8}$' AND `dt_realizacao_ultima_contagem` <> '00000000' AND `dt_lancamento` <> '00000000' AND `dt_realizacao_ultima_contagem` > `dt_lancamento`) AS contagem_apos_lancamento_aaaammdd,
 COUNT_IF(`dt_lancamento` RLIKE '^[0-9]{8}$' AND `dt_ultima_modificacao` RLIKE '^[0-9]{8}$' AND `dt_ultima_modificacao` <> '00000000' AND `dt_lancamento` <> '00000000' AND `dt_ultima_modificacao` < `dt_lancamento`) AS modificacao_antes_lancamento_aaaammdd
FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico;

## 19. Amostra ampliada (500 linhas)
Exportação de amostra para conferência; usuários continuam mascarados. Não é extração completa de cenário.

In [0]:
SELECT `num_inventario_fisico`, `num_item_inventario_fisico`, `num_exercicio_fiscal`, `cod_material`, `desc_material`, `tp_categoria_material`, `num_lote`, `cod_centro`, `cod_deposito`, `ind_item_contado`, `cod_diferenca_item_esta_registrada`, `ind_item_recontado`, `ind_item_eliminado`, `cod_contagem_zero`, `tp_estoque_inventario_fisico`, `tp_estoque_especial`, `tp_inventario_rotativo`, `vl_quantidade_contagem_inventario`, `vl_diferenca_inventario_entrada_contagem`, `vl_precos_venda_excluindo_iva`, `vl_precos_venda_iva`, `vl_inserido_externamente_base_venda`, `cod_cliente`, `cod_fornecedor`, `cod_ordem_cliente`, `num_item_pedido_venda`, `num_divisao_remessa`, `ind_material_valor`, `qt_registrada_antes_contagem`, `qt_quantidade`, `sg_medida_basica_material`, `vl_contabil`, `vl_diferenca`, `cod_moeda`, `dt_lancamento`, `dt_realizacao_ultima_contagem`, `dt_ultima_modificacao`, substring(md5(CAST(`cod_usuario_contagem` AS STRING)),1,8) AS `cod_usuario_contagem_hash`, substring(md5(CAST(`cod_modificado` AS STRING)),1,8) AS `cod_modificado_hash`, `num_referencia_inventario_fisico`, `num_material`, `num_ano_material`, `num_item_material`, `num_recontagem`, `cod_motivo_diferenca`, `dateingest`, `yearingest`, `monthingest` FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico ORDER BY `cod_centro`, `num_exercicio_fiscal`, `num_inventario_fisico`, `num_item_inventario_fisico` LIMIT 500;

## 20. Resumo para copiar
Consolide volume, chave candidata, repetição exata e ingestão. A chave real só pode ser escolhida depois de comparar todas as alternativas na seção 4 e validar os resultados SAP.

In [0]:
WITH base AS (SELECT `cod_centro`, `num_exercicio_fiscal`, `num_inventario_fisico`, `num_item_inventario_fisico`, md5(to_json(named_struct('num_inventario_fisico', `num_inventario_fisico`, 'num_item_inventario_fisico', `num_item_inventario_fisico`, 'num_exercicio_fiscal', `num_exercicio_fiscal`, 'cod_material', `cod_material`, 'desc_material', `desc_material`, 'tp_categoria_material', `tp_categoria_material`, 'num_lote', `num_lote`, 'cod_centro', `cod_centro`, 'cod_deposito', `cod_deposito`, 'ind_item_contado', `ind_item_contado`, 'cod_diferenca_item_esta_registrada', `cod_diferenca_item_esta_registrada`, 'ind_item_recontado', `ind_item_recontado`, 'ind_item_eliminado', `ind_item_eliminado`, 'cod_contagem_zero', `cod_contagem_zero`, 'tp_estoque_inventario_fisico', `tp_estoque_inventario_fisico`, 'tp_estoque_especial', `tp_estoque_especial`, 'tp_inventario_rotativo', `tp_inventario_rotativo`, 'vl_quantidade_contagem_inventario', `vl_quantidade_contagem_inventario`, 'vl_diferenca_inventario_entrada_contagem', `vl_diferenca_inventario_entrada_contagem`, 'vl_precos_venda_excluindo_iva', `vl_precos_venda_excluindo_iva`, 'vl_precos_venda_iva', `vl_precos_venda_iva`, 'vl_inserido_externamente_base_venda', `vl_inserido_externamente_base_venda`, 'cod_cliente', `cod_cliente`, 'cod_fornecedor', `cod_fornecedor`, 'cod_ordem_cliente', `cod_ordem_cliente`, 'num_item_pedido_venda', `num_item_pedido_venda`, 'num_divisao_remessa', `num_divisao_remessa`, 'ind_material_valor', `ind_material_valor`, 'qt_registrada_antes_contagem', `qt_registrada_antes_contagem`, 'qt_quantidade', `qt_quantidade`, 'sg_medida_basica_material', `sg_medida_basica_material`, 'vl_contabil', `vl_contabil`, 'vl_diferenca', `vl_diferenca`, 'cod_moeda', `cod_moeda`, 'dt_lancamento', `dt_lancamento`, 'dt_realizacao_ultima_contagem', `dt_realizacao_ultima_contagem`, 'dt_ultima_modificacao', `dt_ultima_modificacao`, 'cod_usuario_contagem', `cod_usuario_contagem`, 'cod_modificado', `cod_modificado`, 'num_referencia_inventario_fisico', `num_referencia_inventario_fisico`, 'num_material', `num_material`, 'num_ano_material', `num_ano_material`, 'num_item_material', `num_item_material`, 'num_recontagem', `num_recontagem`, 'cod_motivo_diferenca', `cod_motivo_diferenca`, 'dateingest', `dateingest`, 'yearingest', `yearingest`, 'monthingest', `monthingest`))) AS hash_linha, `dateingest` FROM prd_procurement.corp_curated.vw_ds_log_mi24_inventario_fisico),
 k AS (SELECT `cod_centro`, `num_exercicio_fiscal`, `num_inventario_fisico`, `num_item_inventario_fisico`, COUNT(*) AS linhas_chave FROM base GROUP BY `cod_centro`, `num_exercicio_fiscal`, `num_inventario_fisico`, `num_item_inventario_fisico`),
 h AS (SELECT hash_linha, COUNT(*) AS linhas_hash FROM base GROUP BY hash_linha),
 b AS (SELECT COUNT(*) AS linhas, MIN(dateingest) AS primeira_ingestao, MAX(dateingest) AS ultima_ingestao FROM base)
SELECT b.linhas, 'cod_centro + num_exercicio_fiscal + num_inventario_fisico + num_item_inventario_fisico (CANDIDATA)' AS chave_testada,
 (SELECT COUNT(*) FROM k) AS chaves_distintas, (SELECT COUNT_IF(linhas_chave > 1) FROM k) AS chaves_repetidas,
 (SELECT COALESCE(SUM(linhas_hash - 1),0) FROM h) AS repeticoes_exatas,
 b.primeira_ingestao, b.ultima_ingestao FROM b;

## Checklist final para comparação com SAP

- [ ] Anotar data/hora da execução e verificar comentário da view na seção 1, incluindo se aplica filtros da tela MI24.
- [ ] Confirmar chave real: 1,0 linha/chave, sem nulos, e separar duplicatas exatas de versões.
- [ ] Conferir centros 4128 e 4014, volume e formato exato do filtro antes de gerar cenários.
- [ ] Investigar colunas 100% nulas não declaradas “sem fonte”; conferir ausências de referência e origem SAP.
- [ ] Conciliar com o SAP no mesmo dia, mesmos filtros (centro, exercício, depósito, tipo de estoque, status); se a view não aplicar filtros da tela, comparar base completa e equivalente à transação separadamente.
- [ ] Normalizar zeros à esquerda e datas; conferir unidade de medida, moeda, separador decimal e tolerância de 0,005 em `double`.
- [ ] Exportar HTML ou `.ipynb` executado antes de criar notebooks de cenário; nenhum volume esperado foi presumido.
